# Cropping Hands Out

Detects people/hands in leaf photos with YOLOv11 and crops them away, keeping the larger leaf region. Images with no detected hand are copied through unchanged.

In [ ]:
import cv2
import os
from ultralytics import YOLO

model = YOLO('yolo11m.pt') 

def clean_dataset_hands(input_path, output_path):
    os.makedirs(output_path, exist_ok=True)
    
    total_images = 0
    hands_detected = 0
    
    files = [f for f in os.listdir(input_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    total_files = len(files)
    
    print(f"Starting Final Clean on {total_files} images...")
    print("-" * 40)

    for f in files:
        img_path = os.path.join(input_path, f)
        img = cv2.imread(img_path)
        if img is None: 
            continue
        
        total_images += 1
        h, w, _ = img.shape
        
        results = model(img, classes=[0], conf=0.25, imgsz=1280, verbose=False)[0]
        
        final_output = img

        if len(results.boxes) > 0:
            hands_detected += 1
            
            box = results.boxes.data[0].cpu().numpy()
            x1, y1, x2, y2 = map(int, box[:4])
            
            padding = 130
            center_x = (x1 + x2) / 2
            
            if center_x > w / 2:
                final_output = img[0:h, 0:max(0, x1 - padding)]
            else:
                final_output = img[0:h, min(w, x2 + padding):w]

        save_path = os.path.join(output_path, f)
        cv2.imwrite(save_path, final_output)
        
        if total_images % 20 == 0 or total_images == total_files:
            print(f"Progress: {total_images}/{total_files} images cleaned...")

    print("-" * 40)
    print(f"CLEANING COMPLETE")
    print(f"Total Files Processed: {total_images}")
    print(f"Total Hands Excised: {hands_detected}")
    print(f"Clean Dataset Location: {output_path}")

input_folder = r"D:\Desktop\Sesame Datasets\Roboflow (Essa Ahmed)\Bacteria Blight Infected"
output_folder = r"D:\Desktop\Bacteria"

clean_dataset_hands(input_folder, output_folder)